# torch.scan with XNNPACK Backend

This notebook demonstrates exporting a model that uses `torch.scan` (a higher-order op for sequential accumulation) with a linear layer in the combine function, lowering it to the XNNPACK backend, and validating that the ExecuTorch output matches eager PyTorch execution.

Prerequisites: install ExecuTorch with `./install_executorch.sh`.

In [ ]:
import torch
from torch import nn
from torch._higher_order_ops.scan import scan


class ScanLinearModel(nn.Module):
    """A simple RNN-like model using scan with a linear layer."""

    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 4)

    def forward(self, xs):
        def combine_fn(carry, x):
            new_carry = self.linear(carry + x)
            return new_carry, new_carry + 0

        init = torch.zeros_like(xs[0])
        return scan(combine_fn, init, xs)


model = ScanLinearModel().eval()
example_inputs = (torch.randn(5, 4),)
final_carry, outputs = model(*example_inputs)
print(f"final_carry shape: {final_carry.shape}")
print(f"outputs shape:     {outputs.shape}")

In [ ]:
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower

exported = torch.export.export(model, example_inputs)

et_program = to_edge_transform_and_lower(
    exported,
    partitioner=[XnnpackPartitioner()],
).to_executorch()

pte_path = "scan_model.pte"
with open(pte_path, "wb") as f:
    f.write(et_program.buffer)

print(f"Saved to {pte_path}")

In [ ]:
from executorch.runtime import Runtime

runtime = Runtime.get()
program = runtime.load_program(pte_path)
method = program.load_method("forward")

xs = torch.randn(5, 4)
et_carry, et_outputs = method.execute([xs])
eager_carry, eager_outputs = model(xs)

carry_match = torch.allclose(et_carry, eager_carry, rtol=1e-3, atol=1e-5)
outputs_match = torch.allclose(et_outputs, eager_outputs, rtol=1e-3, atol=1e-5)
print(f"carry match:   {carry_match}")
print(f"outputs match: {outputs_match}")

In [ ]:
import os

os.remove(pte_path)
print("Cleaned up")